# 3D toric code — vertical line at $h_x=0$: $O_{FM}$ + Rényi-2 transition

Locates the 2nd-order topological$\to$trivial transition (QMC $h_z^c\!\approx\!0.197$)
from two independent local order parameters, extrapolated to $L\to\infty$.

* **$O_{FM}$** — Fredenhagen–Marcu string ratio on a **fixed $R=1$ loop** (perimeter-4
  plaquette), electric sector, `results/phase_hx0.0_bulkR1/`.
* **$S_2$** — Rényi-2 entropy of the central 4-qubit plaquette, `results/phase_hx0.0_s2plaq/`.

Both are fit with a **symmetric tanh** baseline and an **asymmetric Richards** (5-param
generalized logistic) curve; the inflection is the pseudo-critical $h_z(L)$. Per-point
errors are inflated by the **PDG scale-factor method** ($s=\sqrt{\chi^2/\nu}$) to absorb
seed-to-seed NQS training scatter before the finite-size extrapolation
$h_z(L)=h_z^\infty + b\,(1/L)^x$.

Sections: **1** config · **2** data library · **3** curves + derivatives ·
**4** fit models · **5** per-$L$ fits · **6** honest errors · **7** finite-size extrapolation.

In [ ]:
# ====================== 1 · CONFIG — the one cell to edit ======================
import json, glob, os
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# ---- global plotting style: open "L-shaped" axes, faint grid, consistent DPI ----
plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,   # open "L-shaped" axes, no box
    "axes.grid": True,
    "grid.alpha": 0.3,            # faint grid, never dominant
})
FIGDIR = "figures"                          # gitignored; savefig target (commented until final)
os.makedirs(FIGDIR, exist_ok=True)          # ensure it exists so savefig(...) works when uncommented

ROOT   = "/Users/sanzhar123/Desktop/Approximate-Symmetries-TC-main/results"
FM_DIR = f"{ROOT}/phase_hx0.2_bulkR1"      # O_FM, fixed R=1 loop  (fm_L*.json)
S2_DIR = f"{ROOT}/phase_hx0.2_s2plaq"      # Renyi-2 central plaquette (s2_L*.json)
ENERGY_DIR = f"{ROOT}/phase_hx0.2_energy"  # local observables M_z, A_v  (energy_L*.json)
FIELD  = "hz"                              # swept field label (this line: h_z at h_x=0)

# observables to analyze; O_FM/S2 = order params, M_z/A_v = local (available past where
# O_FM/S2 extraction stops). Trim freely — missing ones are auto-dropped in §2.
OBS = ["O_FM", "S2"]

# ---- RANGE KNOB: only points with H_WINDOW[0] <= h <= H_WINDOW[1] enter the fits ----
H_WINDOW = (0.151, 0.55)                     # e.g. tighten to (0.20, 0.35) around the crossing
EXCLUDE  = {}                               # {(obs, L): [h, ...]} drop specific bad points
#           obs in {"O_FM", "S2"}

# ---- fits ----
FORMS   = ("tanh", "richards")              # baseline first, asymmetric second
N_BOOT  = 500                              # bootstrap resamples for honest h_c(L) errors

# ---- finite-size scaling:  h_c(L) = h_c(inf) + b*(1/L)**x ----
FSS_LS    = None                            # subset of L, e.g. [5, 6, 7]; None = all
FSS_FIX_X = None                            # pin x (e.g. 2.0 = 1/nu_4DIsing); None = fit x free
X_BATTERY = (1.0, 1.50, 2.0)                # exponents swept for the systematic spread
X_BOUNDS  = (1.0, 3.0)                       # physical cap on the FREE-x FSS fit

H_C_QMC = 0.197                             # thermodynamic-limit reference (QMC, hx=0)
rng = np.random.default_rng(0)
print(f"field = {FIELD}   window = {H_WINDOW}   forms = {FORMS}   N_BOOT = {N_BOOT}")

## 2 · Data library

Every point from both campaigns, loaded into a single nested dict `DATA[obs][L]` with
`h`, `y`, `ye` arrays (`raw` keeps the full JSON record). Query it freely in later cells —
e.g. `DATA["O_FM"][6]["y"]` or `points("S2", 7)`.

In [ ]:
# ====================== 2 · DATA LIBRARY ======================
_SRC = {  # obs -> (directory, value_key, err_key)
    "O_FM": (FM_DIR,     "O",   "Oe"),
    "S2":   (S2_DIR,     "S2",  "S2e"),
    "M_z":  (ENERGY_DIR, "mz",  "mz_err"),
    "A_v":  (ENERGY_DIR, "A_v", "A_v_err"),
}
LABEL = {"O_FM": r"$O_{FM}$  (R=1)", "S2": r"$S_2$  (central plaquette)",
         "M_z": r"$\langle M_z\rangle$", "A_v": r"$\langle A_v\rangle$"}

def _load(directory, val_key, err_key):
    recs = {}
    for jp in sorted(glob.glob(os.path.join(directory, "*.json"))):
        d = json.load(open(jp))
        if val_key not in d:
            continue
        recs[int(d["L"])] = dict(
            h=np.array(d["field"], float),
            y=np.array(d[val_key], float),
            ye=np.array(d.get(err_key, np.zeros(len(d["field"]))), float),
            raw=d)
    return recs

DATA = {}
for obs in OBS:
    recs = _load(*_SRC[obs]) if obs in _SRC else {}
    if recs:
        DATA[obs] = recs
    else:
        print(f"[skip] {obs}: no data in {_SRC.get(obs, ('?',))[0]}")
OBS = [o for o in OBS if o in DATA]           # keep only the available observables
print(f"analyzing: {OBS}")

def window_mask(h, obs, L):
    keep = (h >= H_WINDOW[0]-1e-9) & (h <= H_WINDOW[1]+1e-9)
    for hx in EXCLUDE.get((obs, L), []):
        keep &= ~np.isclose(h, hx, atol=1e-6)
    return keep

def points(obs, L, windowed=False):
    """(h, y, ye) for one (obs, L); windowed=True applies H_WINDOW + EXCLUDE."""
    d = DATA[obs][L]
    if not windowed:
        return d["h"], d["y"], d["ye"]
    m = window_mask(d["h"], obs, L)
    return d["h"][m], d["y"][m], d["ye"][m]

for obs in DATA:
    Ls = sorted(DATA[obs])
    print(f"[{obs}]  L = {Ls}")
    for L in Ls:
        h, y, ye = points(obs, L)
        hw, *_ = points(obs, L, windowed=True)
        print(f"    L={L}: {len(h):2d} pts (kept {len(hw):2d} in window)  "
              f"{FIELD} in [{h.min():.3g}, {h.max():.3g}]")

# ---- style dictionaries: color == system size (reused in EVERY plot below) ----
_ALL_LS     = sorted({L for o in DATA for L in DATA[o]})
SIZE_COLORS = dict(zip(_ALL_LS, plt.cm.plasma(np.linspace(0, 0.85, max(len(_ALL_LS), 1)))))
#   plasma stops at 0.85 (no washed-out yellow); small L = purple, large L = orange.
OBS_STYLE   = {"O_FM": ("o", "tab:blue"), "S2": ("s", "tab:red"),   # marker == observable,
               "M_z": ("^", "tab:green"), "A_v": ("v", "tab:purple")}  # fill/line == observable

## 3 · Order parameters and their derivatives

Top row: raw $O_{FM}(h)$ and $S_2(h)$ for every $L$ (shaded band = fit window).
Bottom row: finite-difference derivatives — the peak is a model-free estimate of the
pseudo-critical field.

In [ ]:
# ====================== 3 · CURVES + DERIVATIVES ======================
fig, ax = plt.subplots(2, len(OBS), figsize=(6.5*len(OBS), 8.5), squeeze=False)
obs_list = OBS
for j, obs in enumerate(obs_list):
    Ls = sorted(DATA[obs])
    for L in Ls:
        c = SIZE_COLORS[L]                                    # color == system size
        h, y, ye = points(obs, L)
        ax[0, j].errorbar(h, y, yerr=ye, fmt="o-", ms=4, lw=1.2, capsize=2,
                          color=c, label=f"L={L}")
        hm = 0.5*(h[1:] + h[:-1]); d = np.diff(y)/np.diff(h)
        ax[1, j].plot(hm, d, "o-", ms=4, lw=1.2, color=c, label=f"L={L}")
    # for row in (0, 1):
    #     ax[row, j].axvspan(*H_WINDOW, color="0.85", alpha=0.5, zorder=0)   # shaded fit window
        # ax[row, j].axvline(H_C_QMC, ls="--", color="k", lw=1, label=f"QMC = {H_C_QMC}")
    ax[0, j].set(xlabel=f"${FIELD}$", ylabel=LABEL[obs], title=LABEL[obs])
    ax[1, j].set(xlabel=f"${FIELD}$", ylabel=f"d{obs}/d{FIELD}",
                 title="finite-difference derivative (peak = transition)")
ax[0, 0].legend()                                             # one legend only (color == L)
plt.tight_layout(); plt.show()
fig.savefig(f"{FIGDIR}/vline_hz_curves.png", dpi=300, bbox_inches="tight")

## 4 · Fit models

**Symmetric tanh** (4 params) — $O(h)=A+B\tanh\!\big((h-h_c)/w\big)$; $h_c$ is the inflection
directly. Robust, few parameters, but forces point-symmetry about $h_c$.

**Richards / generalized logistic** (5 params) —
$O(h)=A+(K-A)\big[1+e^{-(h-h_0)/w}\big]^{-\nu}$, $\nu>0$ tunes asymmetry ($\nu=1\Rightarrow$
logistic). With $\nu\neq1$ the inflection is **not** $h_0$:
$$h_c = h_0 + w\ln\nu ,$$
which we evaluate exactly (and re-evaluate on every bootstrap resample — the honest way to
propagate its error).

In [ ]:
# ====================== 4 · FIT MODELS ======================
def richards(h, A, K, h0, w, nu):
    z = np.clip(-(h - h0)/w, -700, 700)        # guard exp overflow on the plateaus
    return A + (K - A) / (1.0 + np.exp(z))**nu

def tanh_model(h, A, B, hc, w):
    return A + B*np.tanh((h - hc)/w)

MODELS = {"richards": richards, "tanh": tanh_model}
NPAR   = {"richards": 5,        "tanh": 4}

def inflection(form, p):
    """Pseudo-critical field = inflection of the fitted curve."""
    if form == "richards":
        A, K, h0, w, nu = p
        return h0 + w*np.log(nu)               # exact inflection of the Richards curve
    A, B, hc, w = p
    return hc

def _p0_bounds(form, h, y):
    lo, hi = float(h.min()), float(h.max())
    hmid, span = 0.5*(lo+hi), (hi-lo) or 1.0
    yl, yr = float(y[0]), float(y[-1])
    if form == "richards":
        # A, K are the left/right plateaus (they carry the rise/fall direction); w>0, nu>0
        p0 = [yl, yr, hmid, 0.05*span, 1.0]
        bnds = ([-np.inf, -np.inf, lo-span, 1e-4, 1e-3],
                [ np.inf,  np.inf, hi+span, span,  50.0])
    else:
        p0 = [0.5*(yl+yr), 0.5*(yr-yl), hmid, 0.05*span]
        bnds = ([-np.inf, -np.inf, lo-span, 1e-4],
                [ np.inf,  np.inf, hi+span, span])
    return p0, bnds

def fit_curve(h, y, ye, form, escale=1.0):
    """Fit one windowed curve; return popt, chi2/dof and the inflection h_c."""
    f = MODELS[form]
    e = ye*escale
    use_w = np.all(np.isfinite(e)) and np.all(e > 0)
    kw = dict(sigma=e, absolute_sigma=True) if use_w else {}
    p0, bnds = _p0_bounds(form, h, y)
    popt, pcov = curve_fit(f, h, y, p0=p0, bounds=bnds, maxfev=80000, **kw)
    resid = y - f(h, *popt)
    chi2 = np.sum((resid/e)**2) if use_w else np.sum(resid**2)
    dof = max(1, len(h) - NPAR[form])
    return dict(popt=popt, pcov=pcov, chi2dof=chi2/dof,
                h_c=float(inflection(form, popt)), weighted=use_w)

## 5 · Per-$L$ fits

Both forms fit to the windowed data for every $L$. The dashed vertical is the inflection
$h_c(L)$; the printed $\chi^2/\nu$ is the input to the scale-factor inflation in §6 — values
$\gg1$ signal that neighbouring points miss any smooth curve by many of their own error bars
(the training-scatter symptom).

In [ ]:
# ====================== 5 · PER-L FITS ======================
FITS = {}   # FITS[obs][form][L] = fit dict
for obs in OBS:
    FITS[obs] = {}
    for form in FORMS:
        FITS[obs][form] = {}
        for L in sorted(DATA[obs]):
            h, y, ye = points(obs, L, windowed=True)
            FITS[obs][form][L] = fit_curve(h, y, ye, form)

fig, ax = plt.subplots(len(FORMS), len(OBS), figsize=(6.5*len(OBS), 4.4*len(FORMS)), squeeze=False)
for i, form in enumerate(FORMS):
    for j, obs in enumerate(OBS):
        Ls = sorted(DATA[obs])
        print(f"\n[{obs} / {form}]  {'L':>2} {'h_c':>8} {'chi2/dof':>9} {'width':>8}"
              + ("  nu" if form == "richards" else ""))
        for L in Ls:
            c = SIZE_COLORS[L]                                # color == system size
            fit = FITS[obs][form][L]; h, y, ye = points(obs, L, windowed=True)
            hh = np.linspace(h.min(), h.max(), 400)
            ax[i, j].errorbar(h, y, yerr=ye, fmt="o", ms=4, capsize=2, color=c, label=f"L={L}")
            ax[i, j].plot(hh, MODELS[form](hh, *fit["popt"]), "-", color=c, lw=1.2)
            ax[i, j].axvline(fit["h_c"], ls="--", color=c, lw=0.8, alpha=0.6)
            extra = f"  {fit['popt'][4]:.2f}" if form == "richards" else ""
            w = abs(fit["popt"][3])
            print(f"    {L:>2} {fit['h_c']:>8.4f} {fit['chi2dof']:>9.1f} {w:>8.4f}{extra}")
        # ax[i, j].axvline(H_C_QMC, ls="--", color="k", lw=1, label="QMC")
        ax[i, j].set(xlabel=f"${FIELD}$", ylabel=LABEL[obs], title=f"{form} fit — {obs}")
ax[0, 0].legend(fontsize=8)                                   # one legend only (color == L)
plt.tight_layout(); plt.show()
fig.savefig(f"{FIGDIR}/vline_hz_perL_fits.png", dpi=300, bbox_inches="tight")

## 6 · Honest errors — PDG scale-factor inflation

The quoted per-point $\sigma_i$ capture only Monte-Carlo sampling variance of a *fixed*
trained network. The dominant scatter at $L=6,7$ is **seed-to-seed NQS training variance**
(each $(h,L)$ is an independently optimized state), which the data itself exposes as
$\chi^2/\nu\gg1$. Following the PDG prescription for mutually inconsistent measurements:

1. fit the smooth model with the quoted errors → $\chi^2/\nu$;
2. inflate: $s=\max(1,\sqrt{\chi^2/\nu})$, $\;\sigma_i\to s\,\sigma_i$;
3. bootstrap $h_c(L)$ with the **inflated** errors, relocating the inflection each resample.

The naive (un-inflated) bootstrap error is reported alongside as the "fictional" comparison.
Treat the inflated error as a **floor**, not the truth (training bias can be coherent).

In [ ]:
# ====================== 6 · HONEST ERRORS (scale-factor + bootstrap) ======================
def bootstrap_hc(obs, form, L, escale, B):
    h, y, ye = points(obs, L, windowed=True)
    e = ye*escale
    p0, bnds = _p0_bounds(form, h, y)
    lo, hi = h.min()-0.05, h.max()+0.05
    out = []
    for _ in range(B):
        yb = y + rng.normal(0, e)
        try:
            pb, _ = curve_fit(MODELS[form], h, yb, p0=p0, bounds=bnds,
                              sigma=e, absolute_sigma=True, maxfev=80000)
            hc = inflection(form, pb)
            if lo < hc < hi:
                out.append(hc)
        except Exception:
            pass
    return np.array(out)

HC = {}   # HC[obs][form][L] = dict(h_c, s, chi2dof, err_naive, err_honest)
for obs in OBS:
    HC[obs] = {}
    for form in FORMS:
        HC[obs][form] = {}
        print(f"\n[{obs} / {form}]  {'L':>2} {'h_c':>8} {'chi2/dof':>9} {'s':>5} "
              f"{'err_naive':>10} {'err_honest':>11}")
        for L in sorted(DATA[obs]):
            fit = FITS[obs][form][L]
            s = max(1.0, np.sqrt(fit["chi2dof"]))
            en = bootstrap_hc(obs, form, L, 1.0, N_BOOT)
            eh = bootstrap_hc(obs, form, L, s,   N_BOOT)
            HC[obs][form][L] = dict(
                h_c=fit["h_c"], s=s, chi2dof=fit["chi2dof"],
                err_naive=float(en.std()) if len(en) else np.nan,
                err_honest=float(eh.std()) if len(eh) else np.nan)
            r = HC[obs][form][L]
            print(f"    {L:>2} {r['h_c']:>8.4f} {r['chi2dof']:>9.1f} {r['s']:>5.1f} "
                  f"{r['err_naive']:>10.4f} {r['err_honest']:>11.4f}")

## 7 · Finite-size extrapolation

$h_c(L) = h_c^{\infty} + b\,(1/L)^x$, weighted by the honest $\sigma[h_c(L)]$ from §6.
`FSS_FIX_X` pins $x$ (e.g. $1.59=1/\nu$ for the 3D Ising / toric-code universality class);
`None` fits $x$ freely (ill-constrained with 4 points — reported for completeness). The
`X_BATTERY` sweep gives the **systematic** spread from the exponent choice, the one
systematic that survives error inflation.

In [ ]:
# ====================== 7 · FINITE-SIZE EXTRAPOLATION ======================
def fss(hc_by_L, use_L=None, fix_x=None):
    use_L = sorted(hc_by_L) if use_L is None else sorted(int(x) for x in use_L)
    L = np.array(use_L, float)
    y = np.array([hc_by_L[k]["h_c"]       for k in use_L])
    e = np.array([hc_by_L[k]["err_honest"] for k in use_L])
    w_ok = np.all(np.isfinite(e)) and np.all(e > 0)
    kw = dict(sigma=e, absolute_sigma=True) if w_ok else {}
    if fix_x is None:
        f = lambda LL, h0, b, x: h0 + b*(1.0/LL)**x
        p, c = curve_fit(f, L, y, p0=[y.min(), 0.3, 0.5*(X_BOUNDS[0]+X_BOUNDS[1])],
                         bounds=([-np.inf, -np.inf, X_BOUNDS[0]], [np.inf, np.inf, X_BOUNDS[1]]),
                         maxfev=80000, **kw)
        h0, b, x = map(float, p); xerr = float(np.sqrt(abs(c[2, 2])))
    else:
        f = lambda LL, h0, b: h0 + b*(1.0/LL)**fix_x
        p, c = curve_fit(f, L, y, p0=[y.min(), 0.3], maxfev=80000, **kw)
        h0, b, x, xerr = float(p[0]), float(p[1]), float(fix_x), 0.0
    return dict(h0=h0, h0_err=float(np.sqrt(abs(c[0, 0]))), b=b, x=x, xerr=xerr,
                L=L, y=y, e=e, weighted=w_ok)

RESULT = {}
fig, ax = plt.subplots(len(FORMS), len(OBS), figsize=(6.5*len(OBS), 4.4*len(FORMS)), squeeze=False)
for i, form in enumerate(FORMS):
    for j, obs in enumerate(OBS):
        F = fss(HC[obs][form], use_L=FSS_LS, fix_x=FSS_FIX_X)
        RESULT[(obs, form)] = F
        # systematic spread over the exponent battery
        batt = []
        for xx in X_BATTERY:
            try: batt.append(fss(HC[obs][form], use_L=FSS_LS, fix_x=xx)["h0"])
            except Exception: pass
        syst = 0.5*(max(batt)-min(batt)) if batt else np.nan
        xtag = f"x={F['x']:.2f}" if FSS_FIX_X is not None else f"x free={F['x']:.2f}±{F['xerr']:.2f}"
        print(f"[{obs} / {form}]  h_c(inf) = {F['h0']:.4f} ± {F['h0_err']:.4f} (stat) "
              f"± {syst:.4f} (syst, exponent)   {xtag}   "
              f"offset vs QMC {F['h0']-H_C_QMC:+.4f}")

        invL = 1.0/F["L"]
        grid = np.linspace(0, invL.max(), 200)                 # fit drawn out to 1/L -> 0
        mk, ocol = OBS_STYLE.get(obs, ("o", "0.3"))            # shape+color == observable
        a = ax[i, j]
        for k, Lval in enumerate(F["L"]):                      # each point filled by its size
            a.errorbar(invL[k], F["y"][k],
                       yerr=(F["e"][k] if F["weighted"] else None),
                       fmt=mk, mfc=SIZE_COLORS[int(Lval)], mec="k", mew=0.4, ms=7,
                       ecolor="0.5", capsize=2, zorder=3,
                       label="$h_c(L)$" if k == 0 else "_nolegend_")
        a.plot(grid, F["h0"] + F["b"]*grid**F["x"], "--", color=ocol, lw=1.4, zorder=2)
        a.errorbar(0, F["h0"], yerr=F["h0_err"], fmt=mk, mfc=ocol, mec="k", mew=0.4,
                   ms=9, ecolor="0.5", capsize=4, zorder=4,
                   label=fr"$h_c(\infty)={F['h0']:.4f}\pm{F['h0_err']:.4f}$")
        a.axhline(H_C_QMC, ls="--", color="k", lw=1, label=f"QMC = {H_C_QMC}")
        _yb = np.concatenate([F["y"], [F["h0"], H_C_QMC]])     # tight band around the values
        _pad = 0.15*(np.ptp(_yb) or 1.0)
        a.set_ylim(_yb.min()-_pad, _yb.max()+_pad)
        a.set(xlabel="$1/L$", ylabel="$h_c(L)$", title=f"{obs} / {form}")
        a.legend(fontsize=8, loc="upper left")
plt.tight_layout(); plt.show()
# fig.savefig(f"{FIGDIR}/vline_hz_fss.png", dpi=300, bbox_inches="tight")

print("\n--- summary: h_c(inf) ---")
for (obs, form), F in RESULT.items():
    print(f"  {obs:>5} / {form:<9}: {F['h0']:.4f} ± {F['h0_err']:.4f}")

## 7b · Exponent sweep (Richards $h_c(L)$)

Refit $h_c(L)=h_c^{\infty}+b\,(1/L)^x$ at a **range of fixed exponents**, all from the
same Richards per-$L$ inflections + honest errors (§5–6). Each extrapolation and its
intercept error is drawn on one axis, so the exponent-choice systematic — the dominant
one once per-point errors are inflated — is visible at a glance. Right panel:
$h_c(\infty)\pm\sigma$ versus $x$ for both observables.

In [ ]:
# ====================== 7b · EXPONENT SWEEP (Richards h_c(L)) ======================
# Refit h_c(L)=h_c(inf)+b*(1/L)**x at a RANGE of FIXED exponents, all from the SAME
# Richards per-L inflections + honest errors (§5/§6). Shows how the L->inf
# extrapolation (and its error) drifts with the assumed exponent.
X_SWEEP = (1.0, 1.25, 1.5, 2.0)     # <-- exponents to compare (edit freely)
FORM_X  = "richards"                          # per-L h_c source: "richards" | "tanh"

cols = plt.cm.viridis(np.linspace(0.15, 0.85, len(X_SWEEP)))   # 2nd sequential map == exponent
_nO = len(OBS)
fig, ax = plt.subplots(1, _nO + 1, figsize=(5.0*_nO + 3.5, 4.7),
                       gridspec_kw={"width_ratios": [1]*_nO + [0.85]}, squeeze=False)
ax = ax[0]
sweep = {}
for j, obs in enumerate(OBS):
    hc = HC[obs][FORM_X]
    use_L = sorted(hc) if FSS_LS is None else sorted(int(v) for v in FSS_LS)
    invL = 1.0/np.array(use_L, float)
    y = np.array([hc[k]["h_c"]        for k in use_L])
    e = np.array([hc[k]["err_honest"] for k in use_L])
    grid = np.linspace(0, invL.max(), 200)
    a = ax[j]
    mk_obs, _oc = OBS_STYLE.get(obs, ("o", "0.3"))
    for k, Lv in enumerate(use_L):                             # h_c(L) points filled by size
        a.errorbar(invL[k], y[k], yerr=e[k], fmt=mk_obs, mfc=SIZE_COLORS[int(Lv)],
                   mec="k", mew=0.4, ms=8, ecolor="0.5", capsize=3, zorder=6,
                   label=f"$h_c(L)$ ({FORM_X})" if k == 0 else "_nolegend_")
    sweep[obs] = []
    for x, c in zip(X_SWEEP, cols):
        try:
            F = fss(hc, use_L=FSS_LS, fix_x=x)
        except Exception as ex:
            print(f"[{obs}] x={x}: fit failed ({ex})"); continue
        sweep[obs].append((x, F["h0"], F["h0_err"]))
        a.plot(grid, F["h0"] + F["b"]*grid**x, "--", color=c, lw=1.4, alpha=0.9)
        a.errorbar(0, F["h0"], yerr=F["h0_err"], fmt="D", mfc=c, mec="k", mew=0.4, ms=8,
                   ecolor="0.5", capsize=4, zorder=5,
                   label=fr"$x={x}$: {F['h0']:.4f}$\pm${F['h0_err']:.4f}")
    a.axhline(H_C_QMC, ls="--", color="k", lw=1, label=f"QMC = {H_C_QMC}")
    a.set(xlabel="$1/L$", ylabel="$h_c(L)$", title=f"{obs} — exponent sweep ({FORM_X})")
    a.legend(fontsize=7.5, loc="upper left")

a = ax[_nO]
for obs in OBS:
    mk, col = OBS_STYLE.get(obs, ("o", "0.3"))                 # shape+color == observable
    if not sweep[obs]:
        continue
    xs, h0, er = zip(*sweep[obs])
    a.errorbar(xs, h0, yerr=er, fmt=mk+"-", color=col, capsize=3, label=obs)
a.axhline(H_C_QMC, ls="--", color="k", lw=1, label=f"QMC = {H_C_QMC}")
a.set(xlabel="exponent $x$", ylabel=r"$h_c(\infty)$", title="intercept vs exponent")
a.legend(fontsize=8)
plt.tight_layout(); plt.show()
fig.savefig(f"{FIGDIR}/vline_hz_exponent_sweep.png", dpi=300, bbox_inches="tight")

print(f"h_c(inf) vs exponent   (from {FORM_X} h_c(L), weighted by honest errors)")
for obs in OBS:
    row = "   ".join(f"x={x}: {h:.4f}+/-{e:.4f}" for x, h, e in sweep[obs])
    print(f"  [{obs:>5}]  {row}")

## Next steps

Four independent estimates of $h_z^c$ are now in `RESULT[(obs, form)]` (2 observables ×
2 fit forms), each with a statistical (inflated-bootstrap) and a systematic (exponent-choice)
error. Combining them — e.g. an error-weighted mean, or treating the $O_{FM}$/$S_2$ spread as
a further systematic — is the next decision; the honest errors here are the inputs to that.

## 8 · Manual FSS — enter your own $h_c(L)$
Self-contained. Type your per-$L$ $h_c$ (and optional errors) into `HC_MANUAL` / `HC_ERR`, 
pick the exponents, and run this cell alone: it fits 
$h_c(L)=h_c(\infty)+b\,(1/L)^x$ at each fixed $x$ and plots the extrapolated intercepts.

In [ ]:
# ============ MANUAL FSS — enter your own h_c(L), sweep the exponent ============
# Self-contained: edit the three dicts/tuples below, then run THIS cell alone.
import numpy as np, matplotlib.pyplot as plt
from scipy.optimize import curve_fit

plt.rcParams.update({"figure.dpi": 120, "font.size": 11, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3})

HC_MANUAL = {4: 0.43, 5: 0.401, 6: 0.3739, 7: 0.3397}   # {L: h_c(L)}                <-- EDIT
HC_ERR    = {4: 0.020, 5: 0.010, 6: 0.005, 7: 0.0050}   # {L: err}; set {} = unweighted  <-- EDIT
X_LIST    = (1.0, 1.5, 2.0)                   # exponents to sweep          <-- EDIT
H_REF     = None                                       # reference line (e.g. QMC), or None

Ls = np.array(sorted(HC_MANUAL), float)
y  = np.array([HC_MANUAL[int(L)] for L in Ls])
e  = np.array([HC_ERR.get(int(L), np.nan) for L in Ls]) if HC_ERR else np.full(Ls.size, np.nan)
w_ok = np.all(np.isfinite(e)) and np.all(e > 0)     # weighted fit only if every L has err>0
assert Ls.size >= 2, "need >= 2 L values"

def _fit(x):
    kw = dict(sigma=e, absolute_sigma=True) if w_ok else {}
    f  = lambda L, h0, b: h0 + b*(1.0/L)**x
    p, cov = curve_fit(f, Ls, y, p0=[y.min(), 0.3], maxfev=80000, **kw)
    return float(p[0]), float(np.sqrt(abs(cov[0, 0]))), float(p[1])

res = []
for x in X_LIST:
    try: h0, h0e, b = _fit(x); res.append((x, h0, h0e, b))
    except Exception as ex: print(f"x={x}: fit failed ({ex})")

grid = np.linspace(0, 1.0/Ls.min(), 200)
cols = plt.cm.viridis(np.linspace(0.15, 0.85, max(len(res), 1)))   # sequential map == exponent
fig, ax = plt.subplots(1, 2, figsize=(12, 4.8))
# --- left: h_c(L) vs 1/L, one dashed fit + diamond intercept per exponent ---
ax[0].errorbar(1.0/Ls, y, yerr=(e if w_ok else None), fmt="o", ms=7, color="0.15",
               mec="k", mew=0.4, ecolor="0.5", capsize=3, zorder=6, label="$h_c(L)$ (manual)")
for (x, h0, h0e, b), c in zip(res, cols):
    ax[0].plot(grid, h0 + b*grid**x, "--", color=c, lw=1.4, alpha=0.9)
    ax[0].errorbar(0, h0, yerr=h0e, fmt="D", mfc=c, mec="k", mew=0.4, ms=8, capsize=4,
                   ecolor="0.5", zorder=5, label=fr"$x={x}$: {h0:.4f}$\pm${h0e:.4f}")
if H_REF is not None: ax[0].axhline(H_REF, ls=":", color="k", lw=1, label=f"ref={H_REF}")
ax[0].set(xlabel="$1/L$", ylabel="$h_c(L)$",
          title=f"manual FSS — exponent sweep ({'weighted' if w_ok else 'unweighted'})")
ax[0].legend(fontsize=8, loc="upper left")
# --- right: extrapolated h_c(inf) vs exponent ---
xs  = [r[0] for r in res]; h0s = [r[1] for r in res]; h0es = [r[2] for r in res]
ax[1].errorbar(xs, h0s, yerr=h0es, fmt="s-", color="tab:red", capsize=3)
if H_REF is not None: ax[1].axhline(H_REF, ls=":", color="k", lw=1, label=f"ref={H_REF}")
ax[1].set(xlabel="exponent $x$", ylabel=r"$h_c(\infty)$", title="intercept vs exponent")
if H_REF is not None: ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()
# fig.savefig(f"figures/vline_hz_manual_fss.png", dpi=300, bbox_inches="tight")

syst = 0.5*(max(h0s) - min(h0s)) if len(h0s) >= 2 else np.nan
print("h_c(inf) vs exponent:")
for x, h0, h0e, b in res: print(f"  x={x:<4}: {h0:.4f} +/- {h0e:.4f}")
print(f"\n  spread over exponents (systematic): +/- {syst:.4f}   "
      f"[range {min(h0s):.4f} .. {max(h0s):.4f}]")